In [ ]:
!pip install -q "qrcode[pil]" statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.8 MB/s eta 0:00:00


In [ ]:
"""
external_validation.py  --  Colab-ready. Tests qr_tamper_effnetb0.keras on UNSEEN data WITHOUT retraining.

Label convention (same as training): 1 = real, 0 = fake/tampered; model output > 0.5 => real.

HOW TO RUN IN COLAB (each block = one cell)
-------------------------------------------
Cell 1:
    !pip install -q "qrcode[pil]" statsmodels
    from google.colab import drive; drive.mount('/content/drive')

Cell 2: paste this whole file, edit CONFIG, then run.

PART A  (no camera needed)  : perturbation test on your existing 6,000 test images
PART B  (camera photos)     : put your phone photos in two folders and evaluate them
PART C  (new unseen codes)  : makes brand-new QR codes + tampering the model never saw
"""
import os, glob, random, io, csv
import numpy as np
import tensorflow as tf
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw
from tensorflow.keras.applications.efficientnet import preprocess_input
from statsmodels.stats.proportion import proportion_confint

# ----------------------------- CONFIG -----------------------------
MODEL_PATH    = "/content/drive/MyDrive/qr_dataset/qr_tamper_effnetb0.keras"
REAL_TEST_DIR = "/content/drive/MyDrive/qr_dataset/real_qr_test"
FAKE_TEST_DIR = "/content/drive/MyDrive/qr_dataset/fake_qr_test"
# PART B folders (you create these): photos of untouched printed codes / tampered printed codes
CAM_REAL_DIR  = "/content/drive/MyDrive/qr_camera/real"
CAM_FAKE_DIR  = "/content/drive/MyDrive/qr_camera/tampered"
N_PER_CLASS_PERTURB = 500     # images per class per perturbation (use 3000 for the full set)
SEED = 42
IMG_SIZE = (224, 224)

model = tf.keras.models.load_model(MODEL_PATH)
random.seed(SEED); np.random.seed(SEED)


# ----------------------------- helpers -----------------------------
def find_images(folder):
    out = []
    for ext in ("png", "jpg", "jpeg", "webp", "bmp"):
        out += glob.glob(os.path.join(folder, "**", f"*.{ext}"), recursive=True)
        out += glob.glob(os.path.join(folder, "**", f"*.{ext.upper()}"), recursive=True)
    return sorted(set(out))

def to_batch(pil_imgs):
    arr = np.stack([np.asarray(im.convert("RGB").resize(IMG_SIZE[::-1], Image.BILINEAR), dtype=np.float32)
                    for im in pil_imgs])
    return preprocess_input(arr)          # same EfficientNet preprocessing as training

def predict_probs(pil_imgs, bs=32):
    p = []
    for i in range(0, len(pil_imgs), bs):
        p += model.predict(to_batch(pil_imgs[i:i + bs]), verbose=0).flatten().tolist()
    return np.array(p)

def report(name, y_true, prob):
    y_pred = (prob > 0.5).astype(int)
    n = len(y_true); k = int((y_pred == y_true).sum())
    lo, hi = proportion_confint(k, n, method="beta")
    fake = y_true == 0; real = y_true == 1
    miss = int(((y_pred == 1) & fake).sum())          # tampered called real  (dangerous)
    false_alarm = int(((y_pred == 0) & real).sum())   # real called tampered
    print(f"{name:34s} acc {k}/{n} = {k/n*100:6.2f}% [95% CI {lo*100:.2f}-{hi*100:.2f}] | "
          f"tampered missed {miss}/{int(fake.sum())} | real flagged {false_alarm}/{int(real.sum())}")
    return dict(name=name, n=n, correct=k, acc=k / n, ci_lo=lo, ci_hi=hi,
                tampered_missed=miss, n_fake=int(fake.sum()), real_flagged=false_alarm, n_real=int(real.sum()))


# ----------------------------- perturbations -----------------------------
def jpeg(q):
    def f(im):
        b = io.BytesIO(); im.convert("RGB").save(b, "JPEG", quality=q); b.seek(0); return Image.open(b).convert("RGB")
    return f
def blur(r):      return lambda im: im.filter(ImageFilter.GaussianBlur(r))
def down_up(s):   return lambda im: im.resize((im.width // s, im.height // s), Image.BILINEAR).resize(im.size, Image.BILINEAR)
def rotate(deg):  return lambda im: im.rotate(deg, resample=Image.BILINEAR, expand=False, fillcolor=(255, 255, 255))
def bright(f_):   return lambda im: ImageEnhance.Brightness(im).enhance(f_)
def contrast(f_): return lambda im: ImageEnhance.Contrast(im).enhance(f_)
def noise(sig):
    def f(im):
        a = np.asarray(im.convert("RGB"), dtype=np.float32) + np.random.normal(0, sig, (im.height, im.width, 3))
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    return f
def camera_like(im):   # rotate + perspective-ish shrink + blur + brightness + noise + jpeg
    im = rotate(random.uniform(-6, 6))(im)
    im = bright(random.uniform(0.6, 1.3))(im)
    im = blur(random.uniform(0.5, 1.6))(im)
    im = noise(random.uniform(3, 10))(im)
    return jpeg(random.choice([50, 65, 80]))(im)

PERTURBATIONS = {
    "none (baseline)": lambda im: im,
    "JPEG q70": jpeg(70), "JPEG q40": jpeg(40),
    "Gaussian blur r1": blur(1), "Gaussian blur r2": blur(2),
    "down/up x2": down_up(2), "down/up x4": down_up(4),
    "rotate +5": rotate(5), "rotate -5": rotate(-5),
    "brightness 0.6": bright(0.6), "brightness 1.4": bright(1.4),
    "contrast 0.6": contrast(0.6),
    "noise sigma 10": noise(10), "noise sigma 25": noise(25),
    "camera-like combo": camera_like,
}


# ----------------------------- PART A -----------------------------
def part_a():
    real = find_images(REAL_TEST_DIR); fake = find_images(FAKE_TEST_DIR)
    random.shuffle(real); random.shuffle(fake)
    real, fake = real[:N_PER_CLASS_PERTURB], fake[:N_PER_CLASS_PERTURB]
    imgs0 = [Image.open(p).convert("RGB") for p in real + fake]
    y = np.array([1] * len(real) + [0] * len(fake))
    rows = []
    for name, fn in PERTURBATIONS.items():
        rows.append(report("A: " + name, y, predict_probs([fn(im) for im in imgs0])))
    return rows


# ----------------------------- PART B -----------------------------
def part_b():
    real = find_images(CAM_REAL_DIR); fake = find_images(CAM_FAKE_DIR)
    assert real and fake, f"Put photos in {CAM_REAL_DIR} and {CAM_FAKE_DIR}"
    imgs = [Image.open(p) for p in real + fake]
    y = np.array([1] * len(real) + [0] * len(fake))
    prob = predict_probs(imgs)
    r = report(f"B: camera photos ({len(real)} real, {len(fake)} tampered)", y, prob)
    # list every mistake so you can describe failure cases in the paper
    paths = real + fake
    with open("camera_errors.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["file", "true", "prob_real"])
        for p, t, pr in zip(paths, y, prob):
            if int(pr > 0.5) != t: w.writerow([p, "real" if t else "tampered", f"{pr:.3f}"])
    print("wrong predictions saved to camera_errors.csv")
    return [r]


# ----------------------------- PART C -----------------------------
def make_new_unseen(n=300, out="/content/new_unseen"):
    """New QR codes (random URLs, NOT from your dataset) with tamper types the model never saw:
    sticker-with-pattern, scratch lines, partial erase, logo-like blob. Then camera-like degradation."""
    import qrcode
    from qrcode.constants import ERROR_CORRECT_M
    os.makedirs(out + "/real", exist_ok=True); os.makedirs(out + "/tampered", exist_ok=True)
    def qr_img(url, box):
        q = qrcode.QRCode(error_correction=ERROR_CORRECT_M, border=4, box_size=box); q.add_data(url); q.make(fit=True)
        return q.make_image(fill_color="black", back_color="white").convert("RGB").resize((512, 512), Image.NEAREST)
    def rand_url():
        w = lambda: "".join(random.choice("abcdefghijklmnopqrstuvwxyz0123456789") for _ in range(random.randint(5, 12)))
        return f"https://{w()}.{random.choice(['com','org','net','io'])}/{w()}/{w()}"
    def tamper(im):
        d = ImageDraw.Draw(im); kind = random.choice(["sticker", "scratch", "erase", "blob"])
        x, y = random.randint(80, 360), random.randint(80, 360)
        if kind == "sticker":
            s = random.randint(50, 110); d.rectangle([x, y, x + s, y + s], fill=(230, 230, 230))
            for _ in range(random.randint(3, 10)):
                a, b = random.randint(x, x + s - 8), random.randint(y, y + s - 8); d.rectangle([a, b, a + 8, b + 8], fill=(0, 0, 0))
        elif kind == "scratch":
            for _ in range(random.randint(2, 5)):
                d.line([random.randint(60, 450), random.randint(60, 450), random.randint(60, 450), random.randint(60, 450)],
                       fill=random.choice([(255, 255, 255), (0, 0, 0)]), width=random.randint(3, 9))
        elif kind == "erase":
            s = random.randint(40, 100); d.rectangle([x, y, x + s, y + random.randint(20, 60)], fill=(255, 255, 255))
        else:
            s = random.randint(40, 90); d.ellipse([x, y, x + s, y + s], fill=(20, 20, 20))
        return im
    for i in range(n):
        url = rand_url(); box = random.choice([6, 8, 10])
        camera_like(qr_img(url, box)).save(f"{out}/real/real_{i:04d}.jpg", quality=90)
        camera_like(tamper(qr_img(url, box))).save(f"{out}/tampered/tamp_{i:04d}.jpg", quality=90)
    return out

def part_c(n=300):
    out = make_new_unseen(n)
    real = find_images(out + "/real"); fake = find_images(out + "/tampered")
    y = np.array([1] * len(real) + [0] * len(fake))
    r = report(f"C: new unseen codes+tamper ({len(real)}+{len(fake)})", y,
               predict_probs([Image.open(p) for p in real + fake]))
    return [r]


# ----------------------------- RUN -----------------------------
all_rows = []
all_rows += part_a()
all_rows += part_c(300)
# all_rows += part_b()     # <- uncomment after you put your phone photos in CAM_REAL_DIR / CAM_FAKE_DIR

import pandas as pd
pd.DataFrame(all_rows).to_csv("external_validation_results.csv", index=False)
print("\nSaved external_validation_results.csv  (download it / copy to Drive; it is your evidence for the paper)")

A: none (baseline)                 acc 1000/1000 = 100.00% [95% CI 99.63-100.00] | tampered missed 0/500 | real flagged 0/500
A: JPEG q70                        acc 1000/1000 = 100.00% [95% CI 99.63-100.00] | tampered missed 0/500 | real flagged 0/500
A: JPEG q40                        acc 1000/1000 = 100.00% [95% CI 99.63-100.00] | tampered missed 0/500 | real flagged 0/500
A: Gaussian blur r1                acc 1000/1000 = 100.00% [95% CI 99.63-100.00] | tampered missed 0/500 | real flagged 0/500
A: Gaussian blur r2                acc 963/1000 =  96.30% [95% CI 94.94-97.38] | tampered missed 0/500 | real flagged 37/500
A: down/up x2                      acc 1000/1000 = 100.00% [95% CI 99.63-100.00] | tampered missed 0/500 | real flagged 0/500
A: down/up x4                      acc 960/1000 =  96.00% [95% CI 94.59-97.13] | tampered missed 1/500 | real flagged 39/500
A: rotate +5                       acc 1000/1000 = 100.00% [95% CI 99.63-100.00] | tampered missed 0/500 | real flagged 